# Hybrid CNN+ViT: 10 Research Ideas - Results Analysis

Comprehensive comparison and evaluation of 10 CNN+ViT hybrid architecture variants trained on APTOS 2019 dataset for diabetic retinopathy classification.

Dataset split: Train 80%, Validation 10%, Test 10%

In [ ]:
# Import Required Libraries
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score
import warnings

warnings.filterwarnings('ignore')

# Setup
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

results_dir = Path("results/hybrid_cnn_vit")
print(f"Results directory: {results_dir}")
print(f"Contents: {list(results_dir.glob('*'))}")

## 1. Load All Model Results

In [ ]:
# Load results from all models
def load_model_results(model_name: str):
    """Load metrics and history for a specific model."""
    model_dir = results_dir / model_name
    
    metrics = {}
    history = {}
    
    # Load metrics
    metrics_file = model_dir / "metrics.json"
    if metrics_file.exists():
        with open(metrics_file) as f:
            metrics = json.load(f)
    
    # Load history
    history_file = model_dir / "training_history.json"
    if history_file.exists():
        with open(history_file) as f:
            history = json.load(f)
    
    return metrics, history

# Discover all model directories
model_dirs = [d.name for d in results_dir.iterdir() if d.is_dir() and d.name != '__pycache__']
model_dirs.sort()

print(f"Found {len(model_dirs)} models:")
for i, model in enumerate(model_dirs, 1):
    print(f"  {i}. {model}")

# Load all results
all_metrics = {}
all_histories = {}

for model_name in model_dirs:
    metrics, history = load_model_results(model_name)
    all_metrics[model_name] = metrics
    all_histories[model_name] = history
    
    if metrics:
        print(f"\n{model_name}:")
        print(f"  Accuracy: {metrics.get('accuracy', 'N/A'):.4f}")
        print(f"  F1: {metrics.get('f1_macro', 'N/A'):.4f}")
        print(f"  QWK: {metrics.get('qwk', 'N/A'):.4f}")

## 2. Performance Metrics Comparison Table

In [ ]:
# Create comparison table
comparison_data = []

for model_name in model_dirs:
    metrics = all_metrics.get(model_name, {})
    if metrics:
        comparison_data.append({
            "Model": model_name.replace("Idea_", ""),
            "Accuracy": metrics.get('accuracy', np.nan),
            "F1 (Macro)": metrics.get('f1_macro', np.nan),
            "QWK": metrics.get('qwk', np.nan),
        })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values("QWK", ascending=False).reset_index(drop=True)

print("\nPerformance Comparison (sorted by QWK):\n")
print(comparison_df.to_string(index=False))

# Save to CSV
comparison_df.to_csv(results_dir / "model_comparison_sorted.csv", index=False)
print(f"\n✓ Saved: {results_dir / 'model_comparison_sorted.csv'}")

## 3. Ranking Visualization - Metrics Comparison

In [ ]:
# Ranking visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("10 Models: Performance Rankings", fontsize=14, fontweight='bold')

metrics_to_plot = ["Accuracy", "F1 (Macro)", "QWK"]
colors = plt.cm.viridis(np.linspace(0, 1, len(comparison_df)))

for ax, metric in zip(axes, metrics_to_plot):
    sorted_df = comparison_df.sort_values(metric, ascending=True)
    ax.barh(sorted_df["Model"], sorted_df[metric], color=colors)
    ax.set_xlabel(metric, fontweight='bold')
    ax.set_title(f"{metric} Ranking")
    ax.grid(axis='x', alpha=0.3)
    
    # Add values on bars
    for i, v in enumerate(sorted_df[metric]):
        ax.text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(results_dir / "00_ranking_comparison.png", dpi=150, bbox_inches='tight')
print("✓ Saved: 00_ranking_comparison.png")
plt.show()

## 4. Training Loss Curves Comparison

In [ ]:
# Plot loss curves for all models
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle("Training and Validation Loss Curves (All Models)", fontsize=14, fontweight='bold')
axes = axes.flatten()

for idx, (ax, model_name) in enumerate(zip(axes, model_dirs)):
    history = all_histories.get(model_name, {})
    
    if history:
        train_loss = history.get("train_loss", [])
        val_loss = history.get("val_loss", [])
        
        ax.plot(train_loss, label="Train", alpha=0.8, linewidth=1.5)
        ax.plot(val_loss, label="Val", alpha=0.8, linewidth=1.5)
        ax.set_title(model_name.replace("Idea_", ""), fontsize=10)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(results_dir / "01_loss_curves_comparison.png", dpi=150, bbox_inches='tight')
print("✓ Saved: 01_loss_curves_comparison.png")
plt.show()

## 5. Validation QWK vs Accuracy Evolution

In [ ]:
# Plot validation metrics evolution
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle("Validation QWK and Accuracy Evolution", fontsize=14, fontweight='bold')
axes = axes.flatten()

for idx, (ax, model_name) in enumerate(zip(axes, model_dirs)):
    history = all_histories.get(model_name, {})
    
    if history:
        val_qwk = history.get("val_qwk", [])
        val_acc = history.get("val_acc", [])
        
        ax.plot(val_qwk, label="QWK", alpha=0.8, linewidth=1.5, marker='o', markersize=3)
        ax.plot(val_acc, label="Accuracy", alpha=0.8, linewidth=1.5, marker='s', markersize=3)
        ax.set_title(model_name.replace("Idea_", ""), fontsize=10)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Score")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
        ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(results_dir / "02_validation_metrics_evolution.png", dpi=150, bbox_inches='tight')
print("✓ Saved: 02_validation_metrics_evolution.png")
plt.show()

## 6. Best Model Analysis - Detailed Confusion Matrix

In [ ]:
# Find best model
best_model_idx = comparison_df["QWK"].idxmax()
best_model_name = comparison_df.loc[best_model_idx, "Model"].replace(" ", "_")

# Map back to original directory name
original_model_name = None
for model_dir in model_dirs:
    if best_model_name.replace("_", " ") in model_dir.replace("_", " "):
        original_model_name = model_dir
        break

if original_model_name is None:
    original_model_name = model_dirs[0]

print(f"\nBest Model: {original_model_name}")
print(f"QWK Score: {comparison_df.loc[best_model_idx, 'QWK']:.4f}")

# Load best model metrics
best_metrics = all_metrics.get(original_model_name, {})
cm = np.array(best_metrics.get('confusion_matrix', []))

if cm.size > 0:
    # Plot confusion matrix for best model
    fig, ax = plt.subplots(figsize=(8, 6))
    
    class_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
    
    ax.set_xlabel('Predicted Label', fontweight='bold')
    ax.set_ylabel('True Label', fontweight='bold')
    ax.set_title(f"Confusion Matrix: {original_model_name.replace('Idea_', '')}", fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(results_dir / "03_best_model_confusion_matrix.png", dpi=150, bbox_inches='tight')
    print("✓ Saved: 03_best_model_confusion_matrix.png")
    plt.show()
    
    # Per-class metrics
    print("\nBest Model - Per-Class Metrics:")
    if 'classification_report' in best_metrics:
        report = best_metrics['classification_report']
        for class_idx, class_name in enumerate(class_names):
            if str(class_idx) in report:
                metrics = report[str(class_idx)]
                print(f"  {class_name:12s}: Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}, F1={metrics['f1-score']:.4f}")

## 7. Per-Class Performance Heatmap (All Models)

In [ ]:
# Per-class performance heatmap
class_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
metrics_matrix = np.zeros((len(model_dirs), len(class_names)))

for idx, model_name in enumerate(model_dirs):
    metrics = all_metrics.get(model_name, {})
    if 'classification_report' in metrics:
        report = metrics['classification_report']
        for class_idx in range(len(class_names)):
            if str(class_idx) in report:
                metrics_matrix[idx, class_idx] = report[str(class_idx)].get('f1-score', 0)

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(metrics_matrix, annot=True, fmt='.3f', cmap='YlGn', ax=ax,
            xticklabels=class_names, yticklabels=[m.replace("Idea_", "") for m in model_dirs],
            cbar_kws={'label': 'F1-Score'})

ax.set_xlabel('DR Class', fontweight='bold')
ax.set_ylabel('Model', fontweight='bold')
ax.set_title('Per-Class F1-Score Heatmap (All Models)', fontweight='bold', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(results_dir / "04_per_class_f1_heatmap.png", dpi=150, bbox_inches='tight')
print("✓ Saved: 04_per_class_f1_heatmap.png")
plt.show()

## 8. Summary & Findings

In [ ]:
print("="*70)
print("HYBRID CNN+ViT: 10 RESEARCH IDEAS - COMPREHENSIVE SUMMARY")
print("="*70)

print(f"\n📊 DATASET: APTOS 2019 Diabetic Retinopathy")
print(f"   Train: 80% | Validation: 10% | Test: 10%")

print(f"\n🏆 TOP 5 MODELS (by QWK):")
top5 = comparison_df.head(5)
for rank, (_, row) in enumerate(top5.iterrows(), 1):
    print(f"   {rank}. {row['Model']:40s} QWK={row['QWK']:.4f}  Acc={row['Accuracy']:.4f}  F1={row['F1 (Macro)']:.4f}")

print(f"\n⚠️  LOWEST 3 MODELS:")
bottom3 = comparison_df.tail(3)
for rank, (_, row) in enumerate(bottom3.iterrows(), len(comparison_df)-2):
    print(f"   {rank}. {row['Model']:40s} QWK={row['QWK']:.4f}  Acc={row['Accuracy']:.4f}  F1={row['F1 (Macro)']:.4f}")

print(f"\n📈 STATISTICS:")
print(f"   Average QWK: {comparison_df['QWK'].mean():.4f}")
print(f"   Std Dev QWK: {comparison_df['QWK'].std():.4f}")
print(f"   Max QWK: {comparison_df['QWK'].max():.4f}")
print(f"   Min QWK: {comparison_df['QWK'].min():.4f}")

print(f"\n💾 OUTPUT FILES:")
output_files = list(results_dir.glob("*.png")) + list(results_dir.glob("*.csv"))
for f in sorted(output_files)[:10]:
    print(f"   - {f.name}")

print("\n" + "="*70)